In [ ]:
import os
import pandas as pd


TRANS_PATH = "LI-Small_Trans.csv"
PATTERNS_PATH = "LI-Small_Patterns.txt"


OUTPUT_TRANS_PATH = "prepared_transactions.csv"
OUTPUT_ACCOUNTS_PATH = "prepared_accounts.csv"



def prepare(trans_path: str, patterns_path: str, output_trans_path: str, output_accounts_path: str):
    
    
    trans = pd.read_csv(trans_path)

    
    trans = trans.rename(columns={
        'Account': 'from_account',
        'Account.1': 'to_account',
        'From Bank': 'from_bank',
        'To Bank': 'to_bank',
        'Amount Paid': 'amount',
        'Payment Currency': 'currency',
        'Payment Format': 'payment_format',
        'Is Laundering': 'is_laundering'
    })

    print(f"Total raw transactions loaded: {len(trans):,}")

    
    pattern_tx_ids = set()
    
    
    if os.path.exists(patterns_path):
        with open(patterns_path, 'r') as f:
            for line in f:
                line = line.strip()
                # Skip empty lines or comment headers
                if not line or line.startswith('#'):
                    continue
                # Parse numeric transaction IDs out of the pattern file
                tokens = line.split(',')
                for token in tokens:
                    clean_token = token.strip()
                    if clean_token.isdigit():
                        pattern_tx_ids.add(int(clean_token))
        print(f"Parsed {len(pattern_tx_ids):,} pattern-indexed transaction IDs.")
    else:
        print(f"Warning: {patterns_path} not found in path. Proceeding without pattern mapping.")

    
    
    laundering = trans[trans['is_laundering'] == 1].copy()
    legitimate = trans[trans['is_laundering'] == 0].copy()

    print(f"Total laundering transactions found: {len(laundering):,}")

    
    target_legit_count = len(laundering) * 3
    legitimate_sampled = legitimate.sample(n=target_legit_count, random_state=42)

    print(f"Legitimate transactions sampled: {len(legitimate_sampled):,}")

    
    prepared_trans = pd.concat([laundering, legitimate_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Final sampled dataset size: {len(prepared_trans):,} rows")

   
    
    from_accts = prepared_trans[['from_account', 'from_bank']].rename(
        columns={'from_account': 'account_id', 'from_bank': 'bank_id'}
    )
    
    to_accts = prepared_trans[['to_account', 'to_bank']].rename(
        columns={'to_account': 'account_id', 'to_bank': 'bank_id'}
    )

    
    accounts = pd.concat([from_accts, to_accts]).drop_duplicates(subset=['account_id']).reset_index(drop=True)

    
    suspicious_accts = set(laundering['from_account']).union(set(laundering['to_account']))
    accounts['is_suspicious'] = accounts['account_id'].isin(suspicious_accts).astype(int)

    print(f"Extracted {len(accounts):,} unique account nodes.")

    
    prepared_trans.to_csv(output_trans_path, index=False)
    accounts.to_csv(output_accounts_path, index=False)

    
    print(f"Saved filtered transactions to: {output_trans_path}")
    print(f"Saved unique accounts list to: {output_accounts_path}")



if __name__ == "__main__":
    prepare(
        trans_path=TRANS_PATH,
        patterns_path=PATTERNS_PATH,
        output_trans_path=OUTPUT_TRANS_PATH,
        output_accounts_path=OUTPUT_ACCOUNTS_PATH
    )

In [ ]:
import pickle
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import seaborn as sns
from pyvis.network import Network


def build_graph(data_path: str, output_path: str):
  df = pd.read_csv(data_path)

  
  G = nx.DiGraph()

  for _, row in df.iterrows():
    src = row['from_account']
    dst = row['to_account']
    is_laundering = int(row['is_laundering'])

    
    if G.has_edge(src, dst):
      G[src][dst]['weight'] += row['amount']
      G[src][dst]['count'] += 1
      if is_laundering:
        G[src][dst]['is_laundering'] = 1
    else:
      G.add_edge(
          src,
          dst,
          weight=row['amount'],
          count=1,
          is_laundering=is_laundering,
          currency=row['currency'],
          payment_format=row['payment_format'],
      )

    
    for node in [src, dst]:
      if node not in G.nodes:
        G.add_node(node)
      if is_laundering:
        G.nodes[node]['is_laundering'] = 1
      elif 'is_laundering' not in G.nodes[node]:
        G.nodes[node]['is_laundering'] = 0

  
  bank_map = df.set_index('from_account')['from_bank'].to_dict()
  bank_map.update(df.set_index('to_account')['to_bank'].to_dict())
  for node in G.nodes:
    G.nodes[node]['bank'] = bank_map.get(node, 'Unknown')

  print(f"Nodes : {G.number_of_nodes():,}")
  print(f"Edges : {G.number_of_edges():,}")

  
  suspicious_nodes = [
      n for n, d in G.nodes(data=True) if d.get('is_laundering') == 1
  ]
  suspicious_edges = [
      (u, v) for u, v, d in G.edges(data=True) if d.get('is_laundering') == 1
  ]
  print(f"Suspicious nodes: {len(suspicious_nodes):,}")
  print(f"Suspicious edges: {len(suspicious_edges):,}")

  
  with open(output_path, 'wb') as f:
    pickle.dump(G, f)
  print(f"\nSaved graph to {output_path}")
  return G



G = build_graph(
    data_path='prepared_transactions.csv', output_path='aml_graph.pkl'
)

In [ ]:
def run_analysis(data_path: str, patterns_path: str):
  df = pd.read_csv(data_path)

  
  laundering_df = df[df['is_laundering'] == 1]
  b2b_pivot = pd.crosstab(
      laundering_df['from_bank'],
      laundering_df['to_bank'],
      values=laundering_df['amount'],
      aggfunc='count',
  ).fillna(0)
  print(b2b_pivot.head())

  
  pay_dist = (
      pd.crosstab(
          df['payment_format'], df['is_laundering'], normalize='columns'
      )
      * 100
  )
  pay_dist.columns = ['Legitimate (%)', 'Laundering (%)']
  print(pay_dist)

  
  typologies = {}
  with open(patterns_path, 'r') as f:
    current_typology = 'General'
    for line in f:
      line = line.strip()
      if line.startswith('#'):
        current_typology = line.replace('#', '').strip()
        if current_typology not in typologies:
          typologies[current_typology] = 0
      elif line:
        typologies[current_typology] = typologies.get(current_typology, 0) + 1

  typology_df = pd.DataFrame(
      list(typologies.items()), columns=['Typology', 'Count']
  )
  print(typology_df)

  return b2b_pivot, pay_dist, typology_df


b2b_pivot, pay_dist, typology_df = run_analysis(
    data_path='prepared_transactions.csv', patterns_path='LI-Small_Patterns.txt'
)

In [ ]:
from IPython.display import IFrame
from pyvis.network import Network

def build_interactive_visual(G, output_html='aml_network_500.html'):
    sub_nodes = list(G.nodes())[:500]
    subG = G.subgraph(sub_nodes)

    net = Network(
        height='750px',
        width='100%',
        bgcolor='#222222',
        font_color='white',
        directed=True,
        notebook=True,
        cdn_resources='in_line'
    )
    net.from_nx(subG)

    
    for node in net.nodes:
        node_id = node['id']
        is_suspicious = G.nodes[node_id].get('is_laundering', 0)
        bank = G.nodes[node_id].get('bank', 'Unknown')

        node['color'] = '#ff4d4d' if is_suspicious == 1 else '#4d94ff'
        node['size'] = 15 if is_suspicious == 1 else 8
        node['title'] = f"Account ID: {node_id}<br>Bank: {bank}<br>Status: {'SUSPICIOUS' if is_suspicious else 'Legitimate'}"

    for edge in net.edges:
        src, dst = edge['from'], edge['to']
        edge_data = G.get_edge_data(src, dst, default={})
        edge['title'] = f"Amount: ${edge_data.get('weight', 0):,.2f}<br>Format: {edge_data.get('payment_format', 'N/A')}"
        edge['color'] = '#ff4d4d' if edge_data.get('is_laundering', 0) == 1 else '#555555'

    
    html_content = net.generate_html()
    with open(output_html, 'w', encoding='utf-8') as f:
        f.write(html_content)

    
    return IFrame(src=output_html, width='100%', height='750px')


build_interactive_visual(G)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Payment Format Distribution
pay_dist.plot(kind='bar', ax=axes[0], colormap='viridis')
axes[0].set_title('Payment Format: Laundering vs Legitimate')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xlabel('Payment Format')

# Chart 2: Top Laundering Typologies
if not typology_df.empty:
    sns.barplot(
        data=typology_df.head(10),
        x='Count',
        y='Typology',
        ax=axes[1],
        palette='magma'
    )
    axes[1].set_title('Top Extracted Laundering Typologies')

plt.tight_layout()
plt.savefig('aml_analysis_charts.png')
plt.show()

In [ ]:
import sys

!{sys.executable} -m pip install neo4j

In [ ]:
from neo4j import GraphDatabase
import pandas as pd

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password123")


trans_df = pd.read_csv('LI-Small_Trans.csv')
trans_df = trans_df.fillna('')

print(f"Loaded {len(trans_df)} rows. Connecting to Neo4j...")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    
    with driver.session() as session:
        # Clear existing graph
        session.run("MATCH (n) DETACH DELETE n")
        

        
        clean_records = []
        for _, row in trans_df.iterrows():
            clean_records.append({
                "from_bank": str(row['From Bank']),
                "to_bank": str(row['To Bank']),
                "amount": float(row['Amount Received']) if row['Amount Received'] != '' else 0.0,
                "payment_format": str(row['Payment Format']),
                "is_laundering": int(row['Is Laundering']) if row['Is Laundering'] != '' else 0
            })

       
        batch_size = 2000
        for i in range(0, len(clean_records), batch_size):
            batch = clean_records[i:i + batch_size]
            
            query = """
            UNWIND $batch AS row
            MERGE (a:Account {id: row.from_bank})
            MERGE (b:Account {id: row.to_bank})
            CREATE (a)-[t:TRANSACTED {
                amount: row.amount,
                payment_format: row.payment_format,
                is_laundering: row.is_laundering
            }]->(b)
            """
            session.run(query, batch=batch)
            print(f"Processed rows {i} to {min(i + batch_size, len(clean_records))}")

print("Full dataset successfully ingested into Neo4j")